In [1]:
import pandas as pd
import numpy as np

drivers = pd.read_csv("../data/processed/drivers.csv")
races = pd.read_csv("../data/processed/races.csv")
results = pd.read_csv("../data/processed/race_driver_labels.csv")
pit_stops = pd.read_csv("../data/processed/pit_stops.csv")
circuits = pd.read_csv("../data/processed/circuits_clean.csv")

In [2]:
races['date'] = pd.to_datetime(races['date'])

In [11]:
results = results.drop(columns=['circuitId', 'year'], errors='ignore')

df = results.merge(
    races[['raceId', 'year', 'date', 'circuitId']],
    on='raceId',
    how='left'
)

df = df.sort_values('date')

df = df[[
    'raceId',
    'driverId',
    'constructorId',
    'circuitId',
    'year',
    'date',
    'grid',               # qualifying position
    'positionOrder',      # finish position
    'points',
    'avgLapTime_s'        # assuming you created this earlier
]]


In [10]:
print(df.columns)

Index(['raceId', 'driverId', 'constructorId', 'circuitId', 'year', 'date',
       'grid', 'positionOrder', 'points', 'avgLapTime_s'],
      dtype='object')


In [12]:
pit_counts = (
    pit_stops.groupby(['raceId', 'driverId'])
             .size()
             .reset_index(name='pitStopCount')
)

df = df.merge(pit_counts, on=['raceId', 'driverId'], how='left')
df['pitStopCount'] = df['pitStopCount'].fillna(0)

In [14]:
df = df.sort_values('date')

df['driver_form_avg'] = (
    df.groupby('driverId')['positionOrder']
      .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

In [15]:
df['constructor_form_avg'] = (
    df.groupby('constructorId')['points']
      .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

In [16]:
driver_debut = (
    df.groupby('driverId')['year']
      .min()
      .reset_index()
      .rename(columns={'year': 'debut_year'})
)

df = df.merge(driver_debut, on='driverId', how='left')
df['driver_experience'] = df['year'] - df['debut_year']
df['driver_experience'] = df['driver_experience'].clip(lower=0)

In [17]:
df['constructor_points_roll5'] = (
    df.groupby('constructorId')['points']
      .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

In [19]:
df['driver_circuit_avg_finish'] = (
    df.groupby(['driverId', 'circuitId'])['positionOrder']
      .transform(lambda x: x.shift(1).expanding().mean())
)

driver_global_avg = (
    df.groupby('driverId')['positionOrder']
      .transform('mean')
)

df['driver_circuit_avg_finish'] = df['driver_circuit_avg_finish'].fillna(driver_global_avg)

In [20]:
df = df.merge(
    circuits[['circuitId', 'circuitType']],
    on='circuitId',
    how='left'
)

df = pd.get_dummies(df, columns=['circuitType'], drop_first=True)

In [21]:
def era_group(year):
    if year < 2000:
        return "classic"
    elif year < 2014:
        return "v8"
    elif year < 2022:
        return "hybrid"
    else:
        return "ground_effect"

df['era'] = df['year'].apply(era_group)
df = pd.get_dummies(df, columns=['era'], drop_first=True)

In [22]:
df = df.drop(columns=['debut_year'])

df = df.dropna(subset=['positionOrder', 'avgLapTime_s', 'points'])

df = df.dropna(subset=['driver_form_avg', 'constructor_form_avg'])

In [23]:
print(df.isna().sum())
print(df.describe())

raceId                       0
driverId                     0
constructorId                0
circuitId                    0
year                         0
date                         0
grid                         0
positionOrder                0
points                       0
avgLapTime_s                 0
pitStopCount                 0
driver_form_avg              0
constructor_form_avg         0
driver_experience            0
constructor_points_roll5     0
driver_circuit_avg_finish    0
circuitType_street           0
era_ground_effect            0
era_hybrid                   0
era_v8                       0
dtype: int64
             raceId      driverId  constructorId     circuitId          year  \
count  10908.000000  10908.000000   10908.000000  10908.000000  10908.000000   
mean     587.261459    317.105611      44.765768     19.707096   2010.740466   
min        1.000000      1.000000       1.000000      1.000000   1996.000000   
25%      141.000000     16.000000       4.00000

In [24]:
df.to_csv("../data/processed/features_v2.csv", index=False)